## Pipeline de simulación de repertorios inmunológicos (*ground truth*) con immuneSIM

Se utiliza el paquete **immuneSIM** en R para generar repertorios inmunológicos simulados bajo distintos escenarios biológicos y profundidades de secuenciación. Este enfoque permite modelar de forma controlada la estructura clonal de los repertorios de células B, incluyendo la asignación de genes V(D)J, la generación de secuencias de la región CDR3 y la distribución de abundancias clonales.

A diferencia de los flujos basados en inferencia (SHazaM + Change-O), immuneSIM entrega directamente la estructura del sistema simulado, permitiendo construir un **ground truth basado en abundancias clonales explícitas** para el análisis de diversidad.

### Generación del repertorio simulado

Los repertorios se generan utilizando la función `immuneSIM()`, la cual produce un objeto que contiene información a nivel de secuencia/clon, incluyendo:

* Secuencia nucleotídica (`sequence`)
* Secuencia aminoacídica (`sequence_aa`)
* Genes V, D y J (`v_call`, `d_call`, `j_call`)
* Región CDR3 (`junction`, `junction_aa`)
* Características de recombinación (`np1`, `np2`, `del_v`, `del_d_5`, `del_d_3`, `del_j`)
* Alineamientos V(D)J (`v_sequence_alignment`, `d_sequence_alignment`, `j_sequence_alignment`)
* Eventos de hipermutación somática (`shm_events`)
* Abundancia clonal (`counts`)
* Frecuencia relativa (`freqs`)
* Identificador del repertorio (`name_repertoire`)

### Construcción del archivo *ground truth* (.tsv)

A partir del objeto generado por immuneSIM, se construye una tabla de trabajo que representa el **ground truth del repertorio simulado**, basada en la estructura interna del simulador.

En este enfoque, cada fila corresponde a una secuencia o clon simulado, cuya abundancia está definida por el simulador.

Las variables incluidas en el archivo son:

* `sequence` → secuencia nucleotídica
* `sequence_aa` → secuencia aminoacídica
* `v_call`, `d_call`, `j_call` → asignación de genes V(D)J
* `junction`, `junction_aa` → región CDR3
* `counts` → número de células asociadas a cada secuencia o clon
* `freqs` → frecuencia relativa en el repertorio

### Exportación del archivo

El dataset se exporta en formato `.tsv` utilizando `write.table()`, generando un archivo por cada escenario biológico (Control, Naive, Sangre periférica, Folicular y Extrafolicular) y por cada profundidad de secuenciación (100 a 102 400 secuencias).

### Uso del *ground truth*

Este archivo se utiliza para:

* Calcular directamente métricas de diversidad (riqueza clonal, Shannon, Simpson y otras).
* Analizar el efecto de la profundidad de secuenciación.
* Comparar los resultados con métodos de inferencia clonal basados en SHazaM + Change-O.

### Consideración metodológica

El uso de immuneSIM permite disponer de una estructura clonal simulada con abundancias explícitas (`counts`), lo que facilita la evaluación de métricas de diversidad en un entorno controlado. Esto permite diferenciar claramente entre:

* **Ground truth** (estructura simulada basada en abundancias).
* **Clustering inferido** (Change-O + SHazaM).


## Configuración del entorno reproducible con renv

Se utiliza el paquete **renv** para gestionar un entorno reproducible del proyecto en R, permitiendo aislar y fijar las versiones exactas de los paquetes utilizados.

Este enfoque asegura que los análisis puedan replicarse en otros sistemas sin problemas de compatibilidad.

### Inicialización del entorno

* `renv::init()` → inicializa el proyecto y genera el archivo `renv.lock`, que almacena las versiones de los paquetes utilizados.

### Instalación de paquetes

Se instalan paquetes desde:

* **CRAN**: `dplyr`, `ggplot2`, `viridisLite`, `readr`, `here`
* **Bioconductor**: `Biostrings`, `IRanges`, `GenomicRanges`

Los paquetes se instalan utilizando `install.packages()` y `BiocManager::install()`, según corresponda.

### Control de versiones

* `renv::snapshot()` → guarda las versiones exactas de los paquetes en el archivo `renv.lock`.
* `renv::restore()` → restaura el entorno del proyecto utilizando las versiones registradas en `renv.lock`.

Este flujo garantiza la **reproducibilidad del análisis**, permitiendo ejecutar el proyecto con las mismas versiones de los paquetes en distintos equipos.


## Simulación de repertorios con immuneSIM

Se utiliza el paquete **immuneSIM** para generar repertorios simulados de secuencias de inmunoglobulinas, configurando distintos parámetros que controlan la composición, diversidad y características biológicas de las secuencias.

### Descripción de variables

* **name_repertoire**: nombre del repertorio simulado.
* **number_of_seqs**: número total de secuencias generadas.
* **species**: especie (por ejemplo, humano).
* **receptor**: tipo de receptor (`ig` para BCR, `tr` para TCR).
* **chain**: tipo de cadena (pesada o liviana).
* **verbose**: activa o desactiva los mensajes durante la simulación.

### Parámetros de distribución clonal

* **equal_cc**: define si todos los clonotipos tienen el mismo tamaño.
* **user_defined_alpha**: controla la uniformidad de la distribución clonal; valores más altos generan una mayor desigualdad entre clones.

### Mutaciones somáticas (SHM)

* **shm**: define el modelo de mutación somática. Puede tomar los siguientes valores:

  * `none`: no se simulan mutaciones.
  * `poisson`: mutaciones aleatorias sin sesgo.
  * `data`: mutaciones basadas en perfiles experimentales, con mayor frecuencia en regiones CDR.
  * `naive`: secuencias sin SHM (puede incluir artefactos técnicos).
  * `motif`: mutaciones dirigidas por motivos específicos.

* **shm.prob**: establece la probabilidad de mutación por secuencia (por ejemplo, 15/350 ≈ 15 mutaciones en 350 nucleótidos).

### Parámetros de recombinación V(D)J

* **vdj_noise**: introduce variabilidad en la selección de genes V, D y J (rango de 0 a 1; valores más altos generan mayor aleatoriedad).

### Longitud de CDR3

* **max_cdr3_length / min_cdr3_length**: establecen los límites mínimo y máximo para la longitud de la región CDR3.

En humanos, la longitud típica de la CDR3 de la cadena pesada (IgH) varía entre 10 y 25 aminoácidos, con una mediana aproximada de 15–16 aminoácidos.


In [1]:
library(immuneSIM)
library(Biostrings)
library(dplyr)
library(here) 

Loading required package: BiocGenerics

Loading required package: generics


Attaching package: 'generics'


The following objects are masked from 'package:base':

    as.difftime, as.factor, as.ordered, intersect, is.element, setdiff,
    setequal, union



Attaching package: 'BiocGenerics'


The following objects are masked from 'package:stats':

    IQR, mad, sd, var, xtabs


The following objects are masked from 'package:base':

    Filter, Find, Map, Position, Reduce, anyDuplicated, aperm, append,
    as.data.frame, basename, cbind, colnames, dirname, do.call,
    duplicated, eval, evalq, get, grep, grepl, is.unsorted, lapply,
    mapply, match, mget, order, paste, pmax, pmax.int, pmin, pmin.int,
    rank, rbind, rownames, sapply, saveRDS, table, tapply, unique,
    unsplit, which.max, which.min


Loading required package: S4Vectors

Loading required package: stats4


Attaching package: 'S4Vectors'


The following object is masked from 'package:utils':

    findMatches


The follo

In [2]:
n_seqs <- 102400
scenario <- "E"

sim_repertoire <- immuneSIM(
  name_repertoire = paste0("sim_rep_", n_seqs),
 number_of_seqs = n_seqs,
  species = "hs",
  receptor = "ig",
  chain = "h",
  verbose= TRUE,
  equal_cc = FALSE,
  user_defined_alpha = 2.5,  
  shm = "poisson",
  shm.prob = 30/350,
  vdj_noise = 0,
  max_cdr3_length =17, 
  min_cdr3_length =13,
 
  )


In [ ]:
df_gt <- as.data.frame(sim_repertoire)


In [ ]:
gt_table <- df_gt %>%
  select(
    sequence,
    v_call,
    d_call,
    j_call,
    junction,
    counts,
    freqs
    )

In [ ]:
gt_file <- file.path(
  "/Users/catg/Desktop/SOFIAC/Gitsofia/tesisbioinf-sofia/results/immunesim_ground_truth",
  paste0("gt_", scenario, "_", n_seqs, ".tsv")
)
write.table(
  gt_table,
  file = gt_file,
  sep = "\t",
  quote = FALSE,
  row.names = FALSE
)

message("Guardado en: ", gt_file)

Guardado en: /Users/catg/Desktop/SOFIAC/Gitsofia/tesisbioinf-sofia/results/immunesim_ground_truth/gt_E_800.tsv

